# SoundSense — Kaggle GPU 5-Fold Model Training & Fine-Tuning

This notebook trains and fine-tunes all ESC-50 neural models (`resnet18`, `sresnet`, `vgg16`, `cnngru`) with **free Kaggle GPU (T4 / P100)**.

### Instructions:
1. Under Notebook Settings, enable **GPU T4 x2** or **GPU P100** and enable **Internet**.
2. Run all cells. All checkpoints will be saved into `/kaggle/working/checkpoints/`.
3. Download the `.pt` files and place them into your local `Sound_Recognition/checkpoints/` folder.

In [ ]:
# 1. Verify GPU availability
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Please enable GPU in the Kaggle notebook settings (Accelerator -> GPU)!')

In [ ]:
# 2. Clone repository into /kaggle/working/Sound_Recognition
import os
os.chdir('/kaggle/working')
!git clone -b feat/ast-deployed https://github.com/Snehith259/Sound_Recognition.git /kaggle/working/Sound_Recognition || (cd /kaggle/working/Sound_Recognition && git fetch && git checkout feat/ast-deployed && git pull)
os.chdir('/kaggle/working/Sound_Recognition')
!pip install -q soundfile librosa torchvision torchaudio transformers

In [ ]:
# 3. Run full GPU 5-Fold Cross Validation & Final Deployment Training
!cd /kaggle/working/Sound_Recognition && PYTHONPATH=/kaggle/working/Sound_Recognition python -m eval.train_cv --model all --epochs 60 --batch-size 64 --device cuda --include-final --overwrite

In [ ]:
# 4. Evaluate all held-out test folds with GPU acceleration
!cd /kaggle/working/Sound_Recognition && PYTHONPATH=/kaggle/working/Sound_Recognition python -m eval.eval_esc50_cv_features --model all --device cuda
!cd /kaggle/working/Sound_Recognition && PYTHONPATH=/kaggle/working/Sound_Recognition python -m eval.report

In [ ]:
# 5. Compress trained checkpoints for easy download
!cd /kaggle/working/Sound_Recognition && zip -r /kaggle/working/trained_checkpoints.zip checkpoints/*.pt checkpoints/*.json checkpoints/cv eval/reports eval/REPORT.md
print('Finished! Download /kaggle/working/trained_checkpoints.zip from the Output panel.')